# Tutorial A01: Querying OPERA Sentinel-1 RTC Products via the ASF API

---

## 1. Tutorial Goal

This tutorial demonstrates how to use the [ASF Search API](https://github.com/asfadmin/Discovery-asf_search) and Python to query NASA OPERA Sentinel-1 RTC (Radiometrically Terrain Corrected) products.

By the end of this tutorial you will be able to:

- Define a spatial query region using WKT (Well-Known Text) format
- Apply filters for time range, orbit direction, platform, and processing level
- Retrieve download URLs and metadata (scene name, acquisition time, polarization, S3 links, etc.)
- Use the results as input for batch download and preprocessing with `s1grits` (Tutorial A02)


---

## 2. Environment Setup

This tutorial requires `asf_search` and several geospatial libraries. Because these libraries depend on compiled C/C++ binaries (e.g. `shapely`, `geopandas`, `gdal`), using Conda is strongly recommended to avoid dependency conflicts.

```bash
conda install -n base conda-libmamba-solver
conda env create -f environment.yml --solver=libmamba
conda activate py312_s1grits
python -m ipykernel install --user --name py312_s1grits --display-name "Python (py312_s1grits)"
jupyter notebook
```


---

## 3. ASF Overview

ASF (Alaska Satellite Facility) is a NASA-affiliated data center operated by the University of Alaska Fairbanks. It specializes in the archiving, distribution, and research support of SAR (Synthetic Aperture Radar) data.

Key services:

- **Data hosting**: Sentinel-1, ALOS PALSAR, RADARSAT, SMAP, and more — at Level-0, GRD, SLC, and RTC processing levels
- **Search API**: Python interface `asf_search` and web portal https://search.asf.alaska.edu
- **OPERA support**: ASF is a core partner in the OPERA project, distributing RTC and other analysis-ready products
- **Cloud access**: All ASF data is hosted on AWS (NASA Earthdata Cloud); OPERA products include S3 URIs for direct cloud streaming


---

## 4. OPERA Sentinel-1 RTC Overview

**OPERA (Observational Products for End-users from Remote-sensing Analysis)** is a NASA-led project that generates standardized, analysis-ready data products (ARD) from existing satellite missions including Sentinel-1, Landsat, and Sentinel-2.

The OPERA Sentinel-1 RTC products used in this tutorial have the following characteristics:

- **Analysis-ready**: Backscatter coefficient ($\gamma^0$) is provided directly, with terrain-induced radiometric distortions removed
- **Geometrically precise**: DEM-based correction eliminates foreshortening and layover; data is projected to UTM
- **Standardized**: 30 m spatial resolution, near-global coverage
- **Time-series stable**: Burst-based packaging ensures strict spatial alignment across all acquisitions


---

## 5. Earthdata Authentication (.netrc Setup)

All ASF downloads require a NASA Earthdata account. Set up authentication once using a `.netrc` file.

**Step 1**: Register at https://urs.earthdata.nasa.gov and authorize ASF DAAC access at https://urs.earthdata.nasa.gov/profile (Applications > Alaska Satellite Facility Data Access).

**Step 2**: Create `~/.netrc` (Unix/macOS) or `%USERPROFILE%\.netrc` (Windows):

```
machine urs.earthdata.nasa.gov
  login YOUR_USERNAME
  password YOUR_PASSWORD
```

**Step 3**: Enable `.netrc` in Python:

```python
import asf_search as asf
session = asf.ASFSession()
session.trust_env = True  # Allow reading .netrc and proxy environment config
```


---

## 6. WKT Format (ROI Definition)

WKT (Well-Known Text) is the standard format used in GIS to describe geometric objects such as points, lines, and polygons.

**Quick tip**: You can get a WKT string directly from the ASF Vertex web portal at https://search.asf.alaska.edu — draw a polygon on the map and copy the WKT from the search bar. No shapefile needed.

The example below defines a rectangular polygon over Wuhan University as the Region of Interest (ROI). Coordinates are in longitude/latitude order (EPSG:4326).

**Example ROI (Wuhan, China):**

```
POLYGON((114.3355 30.5214,114.3823 30.5214,114.3823 30.5583,114.3355 30.5583,114.3355 30.5214))
```


In [ ]:
import asf_search as asf
from pprint import pprint

# Define the query region (ROI) using WKT format
# Coordinate order: longitude latitude
# Example ROI: Wuhan, China
test_wkt = "POLYGON((114.3355 30.5214,114.3823 30.5214,114.3823 30.5583,114.3355 30.5583,114.3355 30.5214))"

print(f"Searching archive for ROI: {test_wkt[:50]}...")

# Submit ASF search query
# Note: processingLevel='RTC' is required to target OPERA products specifically
try:
    resp = asf.search(
        intersectsWith=test_wkt,                     # Spatial filter: WKT ROI
        platform=[asf.PLATFORM.SENTINEL1],           # Platform: Sentinel-1
        processingLevel='RTC',                       # Processing level: must be 'RTC' for OPERA
        start="2025-01-01T00:00:00Z",                # Start time (UTC)
        end="2025-12-31T23:59:59Z",                  # End time (UTC)
        maxResults=1000,                             # Maximum number of results
        flightDirection="ASCENDING"                  # Orbit direction: 'ASCENDING' or 'DESCENDING'
    )

    print(f"\nSearch successful. Found {len(resp)} scenes.")

    if len(resp) > 0:
        item = resp[0]  # Use the first result as an example
        props = item.properties

        print("\n=== Sample Scene Metadata ===")
        print(f"Scene name      : {props.get('sceneName')}")
        print(f"File ID         : {props.get('fileID')}")
        print(f"Acquisition time: {props.get('startTime')}")
        print(f"Processing level: {props.get('processingLevel')}")
        print(f"Polarization    : {props.get('polarization')}")
        print(f"Orbit direction : {props.get('flightDirection')}")
        print(f"AWS S3 URLs     : {props.get('s3Urls')}")

        # Count GeoTIFF files in additionalUrls (OPERA band files)
        tif_files = [u for u in props.get('additionalUrls', []) if u.endswith('.tif')]
        print(f"GeoTIFF files   : {len(tif_files)} files")
    else:
        print("No products found. Check your WKT and time range.")

except asf.ASFSearchError as e:
    print(f"Search error: {e}")


---

## 7. Browse Image Visualization

One of the key advantages of `asf_search` is that it returns not only download URLs but also rich metadata including browse image URLs. These allow you to visually preview a scene before downloading the full product.

ASF provides three browse image resolutions for each OPERA product:

- `standard`: Full resolution (suitable for quality assessment)
- `low-res`: Reduced resolution (faster to load, good for low-bandwidth previews)
- `thumbnail`: Thumbnail size (suitable for overview grids)

The helper function `get_rtc_browse(item)` is defined in `s1grits/asf_viewer.py`. It:

- Extracts the browse URL matching the requested resolution
- Downloads and decodes the image as a `PIL.Image` object
- Saves the image to the specified directory using the scene name as the filename


In [ ]:
from pathlib import Path
from IPython.display import display
from s1grits.asf_viewer import get_rtc_browse

# Save browse images to ./tmp/ relative to this notebook
tmp_dir = Path("./tmp")
tmp_dir.mkdir(parents=True, exist_ok=True)

# browse_type options: 'standard', 'low-res', 'thumbnail'
img = get_rtc_browse(item, browse_type='low-res', save_dir=str(tmp_dir))
display(img)


---

## 8. Metadata Exploration

Each result object returned by `asf_search` contains a `properties` dictionary with all available metadata fields. This is the foundation for building automated download and processing pipelines.

By parsing these fields you can:

- Filter results by orbit path number or flight direction
- Extract specific GeoTIFF URLs (e.g. only the VH polarization band)
- Build time-indexed catalogs for time series analysis


In [ ]:
from pprint import pprint

# Print all metadata fields for the first result
if len(resp) > 0:
    print(f"This scene object contains {len(resp[0].properties)} metadata fields:")
    pprint(resp[0].properties)


---

## 8b. Download Verification

The cell below downloads 3 VV-polarization GeoTIFF files to `./tmp/` to verify that your Earthdata `.netrc` authentication is working correctly.

Before running, ensure your `.netrc` file is configured at `~/.netrc` (Unix/macOS) or `%USERPROFILE%\.netrc` (Windows) with the following content:

```
machine urs.earthdata.nasa.gov
  login YOUR_USERNAME
  password YOUR_PASSWORD
```


In [ ]:
import asf_search as asf
from pathlib import Path
from tqdm.auto import tqdm

# Output directory: ./tmp/ relative to this notebook
out_dir = Path("./tmp")
out_dir.mkdir(parents=True, exist_ok=True)
print(f"Download dir: {out_dir.resolve()}")

# Search for OPERA RTC-S1 scenes (same ROI as above)
test_wkt = (
    "POLYGON((114.3355 30.5214,114.3823 30.5214,"
    "114.3823 30.5583,114.3355 30.5583,114.3355 30.5214))"
)

resp = asf.search(
    intersectsWith=test_wkt,
    platform=[asf.PLATFORM.SENTINEL1],
    dataset="OPERA-S1",
    processingLevel="RTC",
    start="2025-12-15T00:00:00Z",
    end="2025-12-31T23:59:59Z",
    flightDirection="ASCENDING",
    maxResults=1000,
)

print(f"Results: {len(resp)}")
if len(resp) == 0:
    raise RuntimeError("No results found. Check ROI, time range, and filters.")

# Extract VV-polarization GeoTIFF URLs from additionalUrls
vv_urls = []
for result_item in resp:
    for u in (result_item.properties.get("additionalUrls") or []):
        if u.lower().endswith("_vv.tif"):
            vv_urls.append(u)

vv_urls = sorted(set(vv_urls))  # deduplicate and sort
print(f"VV GeoTIFF count: {len(vv_urls)}")
if not vv_urls:
    raise RuntimeError("No *_VV.tif found in additionalUrls.")

print("Example URL:", vv_urls[0])

# Configure session to use .netrc credentials
session = asf.ASFSession()
session.trust_env = True  # Allow reading .netrc / proxy environment config

# Download first 3 files for verification
to_download = vv_urls[:3]
print(f"Downloading {len(to_download)} file(s) for verification...")

for url in tqdm(to_download, desc="Downloading VV.tif", unit="file"):
    fname = url.split("/")[-1]
    out_path = out_dir / fname

    if out_path.exists():
        print(f"  Already exists, skipping: {out_path.name}")
        continue

    print(f"  Saving to: {out_path}")
    asf.download_urls(urls=[url], path=str(out_dir), session=session)

print("\nDone.")


---

## 9. ISO XML Metadata Deep Dive

### 9.1 Cloud-Native vs. Traditional Data Distribution

**Traditional format: ESA SAFE (.zip)**

Standard Sentinel-1 Level-1 data (SLC or GRD) from ESA is distributed as SAFE-format ZIP archives. Even to access a single band or inspect metadata, you typically must download and extract the entire multi-GB archive.

**Cloud-native format: NASA OPERA on S3**

OPERA products on ASF are distributed as unpacked, cloud-optimized files. Each component (GeoTIFF bands, ISO XML metadata) is a separate, independently accessible file. This means:

- You can download only the KB-sized `.iso.xml` to inspect data quality before committing to a full download
- You can stream only the VV band without touching VH or the mask
- Lazy loading and cloud-parallel processing are natively supported

### 9.2 Key Information in the ISO XML

The `.iso.xml` file is the product's technical specification. Key fields include:

- **Radiometry**: Output is $\gamma^0$ (gamma nought), normalized to remove terrain-induced brightness variations. Values are in linear power units. To convert to dB: $10 \cdot \log_{10}(\text{value})$.
- **Geometry**: Static tropospheric delay is corrected; wet tropospheric delay is not. Geocoding uses the EGM2008 gravity model.
- **Grid snapping**: All pixel corner coordinates are snapped to integer multiples of 30 m. This guarantees pixel-perfect alignment across all bursts and dates — no resampling needed for mosaicking.

### 9.3 What OPERA Does NOT Do

The XML also clarifies what processing was NOT applied:

- No speckle filtering
- No multi-looking

The data retains full native resolution with speckle noise. Add filtering in your own pipeline if your application requires it.

### 9.4 UTM Zone Auto-Detection

The XML contains the EPSG code for the UTM projection zone. Pattern: `326XX` = Northern Hemisphere Zone XX; `327XX` = Southern Hemisphere Zone XX. For example, `32650` = WGS 84 / UTM zone 50N.


In [ ]:
from s1grits.asf_viewer import parse_opera_iso_xml, get_utm_zone

# 1. Find the ISO XML URL in additionalUrls
xml_url = next(u for u in item.properties['additionalUrls'] if u.endswith('.iso.xml'))

# 2. Parse the XML and set file_identifier as the index
xml_df = parse_opera_iso_xml(xml_url).set_index('file_identifier')

# 3. Display EPSG code and UTM zone
print(xml_df[['epsg_code', 'utm_zone']])
